In [2]:
import geopandas as gpd

shp_path = r"E:\RanhGioi\DBSCL_utm\DBSCL.shp"
gdf = gpd.read_file(shp_path)

print(gdf.crs)


EPSG:32648


In [3]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import os

# ============================
# 1. PATH
# ============================
RICE_FILE = r"E:\DownloadData\ranh_gioi\LandCoverMap\lc_vn_500m_32648_rice_binary.tif"

RRD_SHAPE = r"E:\RanhGioi\DBSH_utm\DBSH.shp"
MKD_SHAPE = r"E:\RanhGioi\DBSCL_utm\DBSCL.shp"

OUT_DIR = r"E:\DownloadData\co2_ban_do\ouput_rice_mask_region"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================
# 2. Load shapefile theo CRS 32648
# ============================
gdf_rrd = gpd.read_file(RRD_SHAPE).to_crs("EPSG:32648")
gdf_mkd = gpd.read_file(MKD_SHAPE).to_crs("EPSG:32648")

regions = {
    "RRD": gdf_rrd,
    "MKD": gdf_mkd
}

# ============================
# 3. Load rice mask gốc
# ============================
with rasterio.open(RICE_FILE) as src:
    rice = src.read(1)
    meta = src.meta.copy()

# ============================
# 4. CẮT THEO VÙNG
# ============================
for region_name, gdf in regions.items():

    geom = [gdf.unary_union.__geo_interface__]

    with rasterio.open(RICE_FILE) as src:
        masked, _ = mask(src, geom, crop=False, filled=False)

    arr = masked[0]

    # chuyển masked array → float + NaN
    arr = np.where(arr.mask, np.nan, arr.data).astype("float32")

    # output path
    out_path = os.path.join(OUT_DIR, f"rice_mask_{region_name}.tif")

    # metadata
    meta.update({
        "count": 1,
        "dtype": "float32",
        "nodata": np.nan
    })

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr, 1)

    print("✔ Saved:", out_path)


✔ Saved: E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_RRD.tif
✔ Saved: E:\DownloadData\co2_ban_do\ouput_rice_mask_region\rice_mask_MKD.tif
